In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.1 Initialization

Before training, every weight is random, and *how* random matters. Too large and
the loss explodes to NaN; too small and nothing flows. The GPT init (small normal,
std 0.02) starts the model near the uniform baseline `ln(vocab)`, exactly where a
fresh model should be.

In [ ]:
import math
from pocketlm import PocketLM, PocketLMConfig, init_weights

torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=20, block_size=16, n_layer=2, n_head=2, n_embd=32)
noise = torch.randint(0, 20, (200,))
xb = noise[:16].unsqueeze(0)
yb = noise[1:17].unsqueeze(0)
model = init_weights(PocketLM(cfg), std=0.02)
_, loss = model(xb, yb)
print("init loss:", round(loss.item(), 3), "  ln(vocab):", round(math.log(20), 3))

A sane init means the first loss is finite and near `ln(vocab)`, not a NaN and not
huge. That is the launchpad every later lesson builds on.

In [ ]:
import math
assert torch.isfinite(loss)
assert loss.item() < 2 * math.log(20)